# 08 - LLM-Enhanced Recommendation System

This notebook reads the config-driven NLP feature file and generates student recommendation reports. Rule-based recommendations remain the source of truth; the LLM only rewrites the evidence into a clear student-facing report.

## 1. Setup and configuration

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

# Allow this notebook to run from either project/ml or project/ml/notebook.
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "src" / "blended_learning").exists() and (candidate / "config" / "config.json").exists():
        PROJECT_ROOT = candidate.resolve()
        src_dir = candidate / "src"
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        break
else:
    raise FileNotFoundError("Could not find project/ml root containing src/ and config/config.json")

from blended_learning.config.settings import settings
from blended_learning.llm.openrouter import OpenRouterStudentRecommender
from blended_learning.utils.notebook import resolve_project_path
from blended_learning.utils.io import read_csv_configured

cfg = settings.llm
io_cfg = cfg["io"]
markdown_cfg = cfg["markdown_export"]

input_csv = resolve_project_path(io_cfg["input_path"], settings.root)
legacy_resume_csv = resolve_project_path(io_cfg["legacy_resume_csv"], settings.root)
clean_output_csv = resolve_project_path(io_cfg["clean_output_csv"], settings.root)
store_csv = resolve_project_path(io_cfg["store_csv"], settings.root)
current_output_csv = resolve_project_path(io_cfg["current_output_csv"], settings.root)

print(f"Project root: {PROJECT_ROOT}")
print(f"Config file: {settings.config_path}")
print(f"Input CSV: {input_csv}")


Project root: /Users/ougi/Documents/Project/Blended-Learning/ml
Config file: /Users/ougi/Documents/Project/Blended-Learning/ml/config/config.json
Input CSV: /Users/ougi/Documents/Project/Blended-Learning/ml/data/processed/student_recommendation_features.csv
/Users/ougi/Documents/Project/Blended-Learning/ml/data/processed/student_recommendation_reports.csv


## 2. Load recommendation feature input

In [2]:
features_df = read_csv_configured(input_csv)
print(f"Recommendation feature dataset shape: {features_df.shape}")
display(features_df.head())

Recommendation feature dataset shape: (570, 13)


,student_id,student_segment,student_segment_label,open_strengths_en,open_challenges_suggestions_en,strength_themes,challenge_themes,strength_tags,challenge_tags,recommendation_tags,segment_default_tags,final_recommendation_tags,rule_based_recommendations
0,e20210686,1,Moderately Engaged (Passive) Learners,Nothing,Nothing,[],[],[],[],[],"[""interaction"", ""learning_support"", ""motivatio...","[""interaction"", ""learning_support"", ""motivatio...","[{""tag"": ""interaction"", ""title"": ""Increase int..."
1,e20241146,2,Highly Engaged (Active) Learners,"Very good, excellent","No big challenge, i’m the best","[""Learning Convenience & Effectiveness""]",[],"[""learning_effectiveness""]",[],"[""learning_effectiveness""]","[""content_access"", ""digital_skill"", ""engagemen...","[""learning_effectiveness""]","[{""tag"": ""learning_effectiveness"", ""title"": ""I..."
2,e20240609,2,Highly Engaged (Active) Learners,Will try hard,Lack of self-discipline,[],"[""Motivation & Self-Discipline""]",[],"[""motivation""]","[""motivation""]","[""content_access"", ""digital_skill"", ""engagemen...","[""motivation""]","[{""tag"": ""motivation"", ""title"": ""Support motiv..."
3,e20240542,1,Moderately Engaged (Passive) Learners,getting more experience,Discipline on daily studying,"[""Learning Convenience & Effectiveness""]","[""Motivation & Self-Discipline""]","[""learning_effectiveness""]","[""motivation""]","[""learning_effectiveness"", ""motivation""]","[""interaction"", ""learning_support"", ""motivatio...","[""learning_effectiveness"", ""motivation""]","[{""tag"": ""learning_effectiveness"", ""title"": ""I..."
4,e20220287,1,Moderately Engaged (Passive) Learners,Student could retrieve the lesson once they wa...,Add Ai assistance,"[""Access to Learning Materials""]",[],"[""content_access""]",[],"[""content_access""]","[""interaction"", ""learning_support"", ""motivatio...","[""content_access""]","[{""tag"": ""content_access"", ""title"": ""Strengthe..."


## 3. Initialize OpenRouter recommender

In [3]:
recommender = OpenRouterStudentRecommender(settings_obj=settings)

print("Model:", recommender.model)
print("Base URL:", recommender.base_url)
print("Prompt path:", recommender.prompt_path)
print("Generation will use:", "OpenRouter API" if recommender.client else "rule-based fallback; no API key found")

Model: openai/gpt-oss-120b:free
Base URL: https://openrouter.ai/api/v1
Prompt path: /Users/ougi/Documents/Project/Blended-Learning/ml/src/blended_learning/llm/prompts/student_recommendation_prompt.txt
Generation will use: OpenRouter API


## 4. Optional clean batch generation

This step can reuse a previous report CSV and write a clean current-run output. Enable or disable it in `config.json` under `llm.generation.clean_batch.enabled`.

In [ ]:
clean_batch_cfg = cfg["generation"]["clean_batch"]

if clean_batch_cfg.get("enabled", True):
    clean_reports_df = recommender.generate_reports_from_csv(
        input_csv=input_csv,
        resume_csv=legacy_resume_csv,
        output_csv=clean_output_csv,
        limit=clean_batch_cfg.get("limit"),
        resume=clean_batch_cfg.get("resume", True),
        save_each_student=clean_batch_cfg.get("save_each_student", True),
        temperature=clean_batch_cfg.get("temperature"),
        max_tokens=clean_batch_cfg.get("max_tokens"),
    )
    print("Clean batch report shape:", clean_reports_df.shape)
else:
    print("Clean batch generation skipped by config.")

Input dataset shape: (570, 14)
Resume file loaded: /Users/ougi/Documents/Project/Blended-Learning/ml/data/processed/student_recommendation_reports.csv
Reusable previous reports: 204
[1/570] Skipping already processed student: e20210686
[2/570] Skipping already processed student: e20241146
[3/570] Skipping already processed student: e20240609
[4/570] Skipping already processed student: e20240542
[5/570] Skipping already processed student: e20220287
[6/570] Skipping already processed student: e20210180
[7/570] Skipping already processed student: e20241245
[8/570] Skipping already processed student: e20221090
[9/570] Skipping already processed student: e20250089
[10/570] Skipping already processed student: e20240950
[11/570] Skipping already processed student: e20240099
[12/570] Skipping already processed student: e20241378
[13/570] Skipping already processed student: e20210635
[14/570] Skipping already processed student: e20240052
[15/570] Skipping already processed student: e20240937
[1

## 5. Incremental generation with master store

This is the recommended production-safe mode. It stores all generated reports in a master store and creates a clean current output matching the latest input file.

In [ ]:
incremental_cfg = cfg["generation"]["incremental"]

if incremental_cfg.get("enabled", True):
    reports_df = recommender.generate_reports_incremental(
        input_csv=input_csv,
        store_csv=store_csv,
        output_csv=current_output_csv,
        limit=incremental_cfg.get("limit"),
        temperature=incremental_cfg.get("temperature"),
        max_tokens=incremental_cfg.get("max_tokens"),
        save_after_each_new_student=incremental_cfg.get("save_after_each_new_student", True),
    )
    print("Current report shape:", reports_df.shape)
else:
    reports_df = pd.DataFrame()
    print("Incremental generation skipped by config.")

## 6. Normalize segment labels in generated CSV files

This removes unstable raw cluster wording from saved report files. The matching rules are stored in `config.json` under `llm.segment_label_fix`.

In [ ]:
label_fix_cfg = cfg["segment_label_fix"]


def normalize_segment_label(label):
    if pd.isna(label):
        return label

    text = str(label).strip()
    lowered = text.lower()

    for rule in label_fix_cfg.get("rules", []):
        if any(term in lowered for term in rule.get("match_any", [])):
            return rule["label"]

    return text


def fix_segment_labels_in_csv(csv_path):
    csv_path = Path(csv_path)
    if not csv_path.exists():
        print(f"File not found: {csv_path}")
        return None

    df = pd.read_csv(csv_path, **io_cfg.get("read_csv_options", {}))
    segment_col = label_fix_cfg["segment_column"]
    student_id_col = label_fix_cfg["student_id_column"]

    if segment_col not in df.columns:
        print(f"No {segment_col!r} column in: {csv_path}")
        return None

    backup_path = csv_path.with_suffix(csv_path.suffix + label_fix_cfg.get("backup_suffix", ".backup_before_label_fix"))
    df.to_csv(backup_path, **io_cfg.get("write_csv_options", {}))
    print(f"Backup saved: {backup_path}")

    df[segment_col] = df[segment_col].apply(normalize_segment_label)

    if label_fix_cfg.get("deduplicate_student_id", True) and student_id_col in df.columns:
        df[student_id_col] = df[student_id_col].astype(str).str.strip()
        df = df.drop_duplicates(
            subset=student_id_col,
            keep=label_fix_cfg.get("deduplicate_keep", "last"),
        )

    df.to_csv(csv_path, **io_cfg.get("write_csv_options", {}))

    print(f"Fixed file saved: {csv_path}")
    print("Shape:", df.shape)
    print("Segment counts:")
    print(df[segment_col].value_counts())
    return df


fixed_dfs = {}
if label_fix_cfg.get("enabled", True):
    path_lookup = {
        "store_csv": store_csv,
        "current_output_csv": current_output_csv,
        "clean_output_csv": clean_output_csv,
        "legacy_resume_csv": legacy_resume_csv,
    }
    for file_key in label_fix_cfg.get("files", []):
        fixed_dfs[file_key] = fix_segment_labels_in_csv(path_lookup[file_key])
else:
    print("Segment label fixing skipped by config.")

## 7. Quick output check

In [ ]:
if current_output_csv.exists():
    current_df = pd.read_csv(current_output_csv, **io_cfg.get("read_csv_options", {}))
    print(f"Generated reports saved to: {current_output_csv}")
    print(f"Rows: {current_df.shape[0]}")
    display(current_df.head())
else:
    print(f"Current output file does not exist yet: {current_output_csv}")

## 8. Export generated reports to Markdown and notebook

This creates a readable Markdown file and a report-only notebook from `student_recommendation_reports_current.csv`.

In [ ]:
markdown_cfg = cfg["markdown_export"]


def make_student_markdown(row, fields):
    student_id = str(row[fields["student_id"]])
    segment = str(row[fields["segment"]])
    tags = str(row[fields["tags"]])
    report_value = row[fields["report"]]
    report = "" if pd.isna(report_value) else str(report_value)

    return f"""## Student ID: `{student_id}`

**Segment:** {segment}

**Recommendation Tags:** `{tags}`

---

{report}
"""


if markdown_cfg.get("enabled", True):
    csv_path = resolve_project_path(markdown_cfg["input_path"])
    out_ipynb_path = resolve_project_path(markdown_cfg["output_ipynb_path"])
    out_md_path = resolve_project_path(markdown_cfg["output_md_path"])
    out_ipynb_path.parent.mkdir(parents=True, exist_ok=True)

    df_reports = pd.read_csv(csv_path, **io_cfg.get("read_csv_options", {}))
    required_columns = markdown_cfg["required_columns"]
    missing_columns = [col for col in required_columns if col not in df_reports.columns]
    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

    title = markdown_cfg.get("title", "Student Recommendation Reports")
    intro_markdown = f"""# {title}

This notebook was generated from:

`{csv_path}`

**Total student records:** {len(df_reports)}

Each section below contains:

- Student ID
- Student segment label
- Final recommendation tags
- LLM-generated recommendation report
"""

    cells_for_report = [{
        "cell_type": "markdown",
        "metadata": {},
        "source": intro_markdown.splitlines(keepends=True),
    }]

    all_markdown_parts = [intro_markdown, "\n\n---\n"]
    fields = markdown_cfg["fields"]

    for _, row in df_reports.iterrows():
        student_markdown = make_student_markdown(row, fields)
        cells_for_report.append({
            "cell_type": "markdown",
            "metadata": {},
            "source": student_markdown.splitlines(keepends=True),
        })
        all_markdown_parts.extend([student_markdown, "\n\n---\n"])

    notebook_payload = {
        "cells": cells_for_report,
        "metadata": markdown_cfg.get("notebook_metadata", {}),
        "nbformat": 4,
        "nbformat_minor": 5,
    }

    with open(out_ipynb_path, "w", encoding="utf-8") as f:
        json.dump(notebook_payload, f, ensure_ascii=False, indent=2)

    with open(out_md_path, "w", encoding="utf-8") as f:
        f.write("\n".join(all_markdown_parts))

    print("Conversion complete.")
    print(f"Notebook saved to: {out_ipynb_path}")
    print(f"Markdown saved to: {out_md_path}")
    print(f"Rows converted: {len(df_reports)}")
else:
    print("Markdown export skipped by config.")